## ML Training

**Lets start with a story**

Suppose shell has `50,000` incident reports. Some are labeled

```text
- Leak

- Gas Detection

- Fire

- Slip/Fall

- Corrosion

- Equipment Failure
```
then; 

Management asks

> Can AI automatically classify every new incident report?

You should immediately answer `Yes`. But then;

`How?`

---

### This brings `Machine Learning`

Definition

> Machine Learning is the process of enabling a computer to discover patterns from historical data in order to make predictions on new, unseen data.

---

`ML flow`;

Historical reports

↓

Model learns

↓

New report

↓

Prediction

---

**Real example**

Historical

```text
Pressure leak detected...

↓

Leak
```

Historical

```text
Worker slipped...

↓

Slip/Fall
```

Historical

```text
Gas detector triggered alarm

↓

Gas Detection
```

`then after learning;`

New report

```text
Gas leakage detected near separator.
```

Model predicts

```text
Gas Detection
```

Nobody wrote

```python
if "gas" in report:
```

The model learned it during training.

---

---

### 1. Supervised Learning

Definition

Students already know the definition and examples of what a `Supervised Learning` is; if you still don't remember visit: https://www.ibm.com/think/topics/supervised-learning#1509394340   *to read more.*

Other: https://aws.amazon.com/compare/the-difference-between-machine-learning-supervised-and-unsupervised/#whats-the-difference-between-supervised-and-unsupervised-machine-learning--atuivl 

---

**Why `split data`?**

Can we train and test using the same data?

If you say `Yes` then you're `wrong`.

`Analogy`:

A teacher gives students exam questions before the exam. Everyone scores 100%.

Did they learn?

`No`.

They memorized. That's exactly the same thing that happens in ML.

---

`Train/Test split`


```text
    Dataset

    ↓

    80%

    Training

    ↓

    20%

    Testing
```

---

Use

```python
from sklearn.model_selection import train_test_split
```

---

Code

```python
X = df["report_text"]

y = df["incident_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

Why random_state? For `reproducibility`.

---

### What is `data leakage`?


Should `TF-IDF` learn using all data? As you may have guessed the answer is `No`. But `why`?

`Because` the test data must remain unseen. 


`Professional pipeline`

```text
    Training Data

    ↓

    Fit TF-IDF

    ↓

    Transform Training

    ↓

    Transform Testing
```

**NOT**

```text
    Entire Dataset

    ↓

    Fit TF-IDF

    ↓

    Split
```

This is one of the biggest mistakes beginners make. Be careful

Learn more: https://www.ibm.com/think/topics/data-leakage-machine-learning

---

### Building the `ML Pipeline`

Now let's update the project.

```
src/

model.py
```

---

Why another `module`? 

Again `single responsibility.`

---

# model.py

```python
from sklearn.linear_model import LogisticRegression


class IncidentClassifier:

    def __init__(self):

        self.model = LogisticRegression(max_iter=1000)

    def train(self, X_train, y_train):

        self.model.fit(X_train, y_train)

    def predict(self, X):

        return self.model.predict(X)
```

**Notice:** The `model` knows nothing about `CSV` cleaning, `EDA`, or `streamlit`. Just `one job`.

---

### Why did we choose `Logistic Regression`?

Why not `Deep Learning`?

That's because professional engineers always start with a simple `baseline`. Read about `baseline` here: https://www.ibm.com/think/topics/logistic-regression#684929715


Baseline

↓

Measure performance

↓

Improve.

Never start with `BERT` (an algorithms used to train NLP models which we'll introduce later).

Other: https://github.com/devmab24/3Logy-NDI-AI-ML-Crash/tree/main/docs/ML_DL_models/Structured_data/Supervised-Learning/Regression/Logistic_Regression

---

### Update `main.py` module

`Workflow`:

```python
    loader = DataLoader()

    processor = TextPreprocessor()

    engineer = FeatureEngineer()

    classifier = IncidentClassifier()
```

`The Pipeline`:

```text
    Load Data

    ↓

    Clean Text

    ↓

    Train/Test Split

    ↓

    TF-IDF

    ↓

    Train Model

    ↓

    Evaluate
```

---

### `Training`

```python
    X_train_features = engineer.fit_transform(X_train)

    X_test_features = engineer.transform(X_test)

    classifier.train(
        X_train_features,
        y_train
    )
```

---

### `Prediction`

```python
    predictions = classifier.predict(
        X_test_features
    )
```

You now have your first NLP AI model built from scratch.

---

### `Evaluation`

Question to ask yourself.

`How good is the model`? i.e `Accuracy`?

Not enough.

---

**Now**;

```python
from sklearn.metrics import accuracy_score
```

```python
accuracy_score(
    y_test,
    predictions
)
```

---

Then  `confusion matrix`. Don't know what a `confusion matrix` is? Visit: https://github.com/devmab24/3Logy-NDI-AI-ML-Crash/blob/main/docs/ML_DL_models/Structured_data/Supervised-Learning/Classification/binary_classification/end-to-end-heart-disease-classification.ipynb

---

Visualization

```
                Predicted

                Leak Fire

    Actual Leak  40    3

    Actual Fire   2   55
```

Now this immediately gives you the understanding of `mistakes`.

---

Then; `classification report`. Don't know what a `classification report` is? Visit: https://github.com/devmab24/3Logy-NDI-AI-ML-Crash/blob/main/docs/ML_DL_models/Structured_data/Supervised-Learning/Classification/binary_classification/end-to-end-heart-disease-classification.ipynb

```python
    from sklearn.metrics import classification_report
    ```

    ```python
    print(
    classification_report(
    y_test,
    predictions
    ))
```

Remember;

- Precision

- Recall

- F1-score

- Support

Note: These are AWS AI Practitioner topics as well.

---

### Saving the `model`.

Remember that professional engineers never retrain every time. The load from a saved model

```
models/
```

---

Use

```python
import joblib
```

---

**Save**

```python
    joblib.dump(
    classifier.model,
    "models/incident_classifier.pkl"
    )
```

---

**Save TF-IDF too**.

People usually forget this.

```python
    joblib.dump(
    engineer.vectorizer,
    "models/vectorizer.pkl"
    )
```

Note: Prediction requires the `same vocabulary`.

---

**Load `model`**

```python
    model = joblib.load(
    "models/incident_classifier.pkl"
    )
```

---

**Predict `new incident`**

```python
    new_report = [
    "Gas leakage detected near separator."
    ]
```

*Transform*

```python
    features = vectorizer.transform(
    new_report
    )
```

*Predict*

```python
    prediction = model.predict(
    features
    )
```

*Output*

```
Gas Detection
```

You now have your first working **AI** application.

---

### Updated project structure

```
    smart_incident_report_analyzer/

    │
    ├── data/
    │
    ├── models/
    │   ├── incident_classifier.pkl
    │   └── vectorizer.pkl
    │
    ├── notebooks/
    |   └── ***
    │
    ├── src/
    │   ├── __init__.py
    │   ├── incident.py
    │   ├── data_loader.py
    │   ├── eda.py
    │   ├── preprocessing.py
    │   ├── feature_engineering.py
    │   ├── model.py
    │   └── utils.py
    │
    ├── train.py
    ├── predict.py
    ├── main.py
    ├── requirements.txt
    ├── README.md
    └── .gitignore
```

---

**New files introduced**

1. `train.py`

Responsible for:

* Loading the dataset.
* Cleaning and preprocessing text.
* Splitting the data.
* Training the vectorizer.
* Training the classifier.
* Evaluating the model.
* Saving the trained artifacts.

This script is run only when you want to create or update the model.

---

2. `predict.py`

Responsible for:

* Loading the saved model.
* Loading the saved TF-IDF vectorizer.
* Accepting new incident reports.
* Applying the same preprocessing pipeline.
* Generating predictions.

Separating training from prediction reflects how production ML systems are built.

---

**End of `phase 4` deliverables**

By the end of this phase, students should have:

* A clean and modular ML project structure.
* A reusable `IncidentClassifier` class.
* A `train.py` script that trains and evaluates the model.
* A `predict.py` script that performs inference on new reports.
* Saved model artifacts (`incident_classifier.pkl` and `vectorizer.pkl`).
* A complete understanding of:

  * Train/test splitting
  * Data leakage
  * TF-IDF fitting vs. transforming
  * Baseline models
  * Model evaluation (Accuracy, Precision, Recall, F1-score)
  * Model persistence with `joblib`

---

---

**`Recommended`** task for you.

Although this uses **Logistic Regression** as the baseline, don't stop here. This allows you to compare multiple classical algorithms on the same dataset:

* **Logistic Regression** (baseline)
* **Naive Bayes** (excellent for text classification)
* **Linear Support Vector Machine (LinearSVC)** (often a top performer for TF-IDF features)
* **Random Forest** (to illustrate why tree-based models are usually less effective on sparse text vectors)
* **Decision Tree** (for comparison and interpretability)

Train with the above listed models, then compare:

| Model | Accuracy | Precision | Recall | F1-score | Training Time |
| ----- | -------- | --------- | ------ | -------- | ------------- |

This exercise reinforces an important engineering principle: **don't assume a model is best; rather measure it.** That mindset is central to both professional ML engineering and the AWS Machine Learning certification path.